# Unpooled Good-School DiD (1 -> 2 good schools, 2km)

This notebook runs a more restrictive school-by-school DiD variant focused on flats that gain the focal school as an additional good school at the `2km` boundary.

The active path again remains unpooled: define a local sample around each focal school, apply a school-specific treatment/control construction, test pre-trends, estimate the DiD, and summarize the results across schools.

In [8]:
import geopandas as gpd
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re
from shapely import wkt
    
# read in HDB data
resale_df = pd.read_csv("../data/processed/final_resale_data_with_names_hawkers.csv")
resale_df['Date'] = pd.to_datetime(resale_df['Date'])
resale_df['geometry'] = resale_df['geometry'].apply(wkt.loads)
resale_gdf = gpd.GeoDataFrame(resale_df, geometry='geometry', crs='EPSG:3414')


## Load Core Inputs

These opening cells load the hawker-augmented resale dataset and the school geometry inputs used in this `2km` unpooled analysis.

The geometry handling is the same as elsewhere in the project: coordinates are projected into EPSG:3414 so distance bands can be interpreted in metres.

In [9]:
resale_gdf.columns

Index(['month', 'town', 'flat_type', 'storey_range', 'floor_area_sqm',
       'flat_model', 'lease_commence_date', 'resale_price', 'remaining_lease',
       'address', 'year', 'latitude', 'longitude', 'geometry', 'Date',
       'num_nearby_malls', 'num_nearby_mrt', 'num_unique_mrt_lines',
       'num_schools_0_1km_polygon', 'num_schools_0_1km_xy',
       'num_schools_1_2km_polygon', 'num_schools_1_2km_xy',
       'num_good_schools_0_1km_polygon', 'num_good_schools_0_1km_xy',
       'num_good_schools_1_2km_polygon', 'num_good_schools_1_2km_xy',
       'good_school_names_0_1km_polygon', 'good_school_names_0_1km_xy',
       'good_school_names_1_2km_polygon', 'good_school_names_1_2km_xy',
       'school_names_0_2km_xy', 'hawker_names_0_1km'],
      dtype='object')

In [2]:
# read in school data and filter by good schools
school_locations_df = gpd.read_file("../data/processed/schools/final_primary_schools_with_school_boundaries.geojson")
good_school_df = pd.read_csv("../data/processed/schools/school_admissions_no_gep_sap.csv")

school_locations_df["good_school"] = school_locations_df["school_name"].isin(good_school_df["School"])

# Ensure schools are in EPSG:3414
school_locations_df = school_locations_df.to_crs('EPSG:3414')
print(school_locations_df.head())

                    school_name  postal_code             X             Y  \
0      ADMIRALTY PRIMARY SCHOOL       738907  24296.625086  47144.770059   
1  AHMAD IBRAHIM PRIMARY SCHOOL       768643  27936.777996  46125.163607   
2                AI TONG SCHOOL       579646  27966.808830  38071.919118   
3      ALEXANDRA PRIMARY SCHOOL       159016  26956.333986  30410.808314   
4   ANCHOR GREEN PRIMARY SCHOOL       544969  33992.445947  41365.632950   

   start_year  end_year                                           geometry  \
0        2014      2026  POLYGON ((24384.885 47241.408, 24367.107 47281...   
1        2014      2026  POLYGON ((27920.039 46227.414, 27919.85 46225....   
2        2014      2026  POLYGON ((28014.157 37989.036, 28059.1 38094.9...   
3        2014      2026  POLYGON ((27097.976 30436.102, 27097.942 30436...   
4        2014      2026  POLYGON ((34094.656 41418.979, 34095.029 41419...   

   good_school  
0        False  
1        False  
2         True  
3     

In [3]:
# calculating location of school using polygon geometry
school_gdf = school_locations_df.copy()
school_gdf["start_year"] = pd.to_numeric(school_gdf["start_year"], errors="coerce")
school_gdf["end_year"] = pd.to_numeric(school_gdf["end_year"], errors="coerce").fillna(np.inf)


good_school_poly_gdf = gpd.GeoDataFrame(
    school_locations_df[school_locations_df["good_school"]].copy(),
    geometry="geometry",
    crs="EPSG:3414"
)

print(good_school_poly_gdf)

                                 school_name  postal_code             X  \
2                             AI TONG SCHOOL       579646  27966.808830   
7              ANGLO-CHINESE SCHOOL (JUNIOR)       227988  28916.316690   
8             ANGLO-CHINESE SCHOOL (PRIMARY)       309918  28225.539585   
24                      CATHOLIC HIGH SCHOOL       579767  29288.969628   
32                  CHIJ PRIMARY (TOA PAYOH)       319765  28949.153729   
33           CHIJ ST. NICHOLAS GIRLS' SCHOOL       569405  28104.019593   
34                            CHONGFU SCHOOL       768959  28666.661305   
53      FAIRFIELD METHODIST SCHOOL (PRIMARY)       139648  22671.351588   
59                   FRONTIER PRIMARY SCHOOL       648200  13127.742393   
71            HOLY INNOCENTS' PRIMARY SCHOOL       536451  34765.903682   
86                           KONG HWA SCHOOL       399772  34124.703658   
88     KUO CHUAN PRESBYTERIAN PRIMARY SCHOOL       579793  30351.430914   
93                       

## Analysis Window And Storage

This section defines the common pre/post time window and creates the result list that will store one unpooled estimate per school.

In [4]:
# ── 3. DEFINE SAMPLE WINDOW ───────────────────────────────────────────────────
pre_start  = pd.Timestamp('2018-09-01')
pre_end    = pd.Timestamp('2021-08-31')
post_start = pd.Timestamp('2021-10-01')
post_end   = pd.Timestamp('2023-03-31')

# ── 4. STORAGE FOR RESULTS ────────────────────────────────────────────────────
results = []

## School-Specific Second-School 2km DiD Loop

This main loop implements the stricter `2km` design. For each focal school, the notebook:
- works around the focal school polygon and uses an explicit outer comparison area for the `2km` design
- tracks focal-school membership at the `1km` and `2km` boundaries before and after the update
- treats flats that stay out of the focal school's `1km` catchment but gain the focal school as an additional good school in the `2km` specification
- keeps controls in the outer band with stable focal-school exposure across periods
- holds normal-school composition fixed
- applies school-specific pre-trend and DiD regressions with the same basic modeling structure as the other unpooled notebooks

Compared with the broader `2km` notebook, this version narrows the treated group to a second-good-school transition within the `2km` specification.

In [12]:
# ── helper: does this row contain the focal school name ────────────────────
def has_focal_school(name_cell, school_name):
    if pd.isna(name_cell):
        return False
    schools = [s.strip().upper() for s in str(name_cell).split('|') if s.strip()]
    return school_name.strip().upper() in schools


# ── 5. LOOP OVER EACH GOOD SCHOOL ────────────────────────────────────────────
for _, school in good_school_poly_gdf.iterrows():
    school_name = school['school_name'].strip().upper()
    print(f"\n{'='*70}")
    print(f"Processing: {school_name}")
    print(f"{'='*70}")

    # -----------------------------------------------------------------------
    # 5a. LOCAL SAMPLE: keep transactions within 3km of focal school polygon
    # -----------------------------------------------------------------------
    school_geom = school['geometry']
    buffer_3km = school_geom.buffer(3000)   # EPSG:3414, so metres
    buffer_2km = school_geom.buffer(2000)
    hdb_local = resale_gdf[resale_gdf.geometry.within(buffer_3km)].copy()

    if hdb_local.empty:
        print("  Skipping: no transactions within 3km.")
        continue

    # -----------------------------------------------------------------------
    # 5b. TIME WINDOW
    # -----------------------------------------------------------------------
    hdb_local = hdb_local[
        ((hdb_local['Date'] >= pre_start) & (hdb_local['Date'] <= pre_end)) |
        ((hdb_local['Date'] >= post_start) & (hdb_local['Date'] <= post_end))
    ].copy()

    if hdb_local.empty:
        print("  Skipping: no transactions in sample window.")
        continue

    hdb_local['post'] = (hdb_local['Date'] >= post_start).astype(int)

    # Explicit outer control band: 2-3km from the focal school polygon.
    hdb_local['in_2_3km_band'] = (
        hdb_local.geometry.within(buffer_3km)
        & ~hdb_local.geometry.within(buffer_2km)
    ).astype(int)

    # -----------------------------------------------------------------------
    # 5c. FOCAL-SCHOOL PRE / POST RING MEMBERSHIP USING SCHOOL-NAME COLUMNS
    # -----------------------------------------------------------------------
    hdb_local['pre_in_0_1km_focal'] = hdb_local['good_school_names_0_1km_xy'].apply(
        lambda x: has_focal_school(x, school_name)
    ).astype(int)

    hdb_local['post_in_0_1km_focal'] = hdb_local['good_school_names_0_1km_polygon'].apply(
        lambda x: has_focal_school(x, school_name)
    ).astype(int)

    hdb_local['pre_in_1_2km_focal'] = hdb_local['good_school_names_1_2km_xy'].apply(
        lambda x: has_focal_school(x, school_name)
    ).astype(int)

    hdb_local['post_in_1_2km_focal'] = hdb_local['good_school_names_1_2km_polygon'].apply(
        lambda x: has_focal_school(x, school_name)
    ).astype(int)

    # DID design:
    # treated  = outside 0-1km and 1-2km pre, inside 1-2km post for THIS focal school
    #            and explicitly not coming from 0-1km
    # control  = in the explicit 2-3km band and outside 0-1km / 1-2km in both pre and post
    did_local = hdb_local[
        (hdb_local['pre_in_0_1km_focal'] == 0) &
        (hdb_local['pre_in_1_2km_focal'] == 0)
    ].copy()

    did_local['treated'] = (
        (did_local['pre_in_0_1km_focal'] == 0) &
        (did_local['pre_in_1_2km_focal'] == 0) &
        (did_local['post_in_0_1km_focal'] == 0) &
        (did_local['post_in_1_2km_focal'] == 1)
    ).astype(int)

    did_local = did_local[
        ((did_local['treated'] == 1) & (did_local['post_in_0_1km_focal'] == 0) & (did_local['post_in_1_2km_focal'] == 1)) |
        ((did_local['treated'] == 0) & (did_local['in_2_3km_band'] == 1) & (did_local['post_in_0_1km_focal'] == 0) & (did_local['post_in_1_2km_focal'] == 0))
    ].copy()

    # -----------------------------------------------------------------------
    # 5d. GLOBAL COUNT VARIABLES (for composition checks)
    # -----------------------------------------------------------------------
    did_local['pre_num_good_schools_0_1km']  = did_local['num_good_schools_0_1km_xy']
    did_local['post_num_good_schools_0_1km'] = did_local['num_good_schools_0_1km_polygon']
    did_local['pre_num_good_schools_1_2km']  = did_local['num_good_schools_1_2km_xy']
    did_local['post_num_good_schools_1_2km'] = did_local['num_good_schools_1_2km_polygon']

    did_local['pre_num_schools_0_1km']  = did_local['num_schools_0_1km_xy']
    did_local['post_num_schools_0_1km'] = did_local['num_schools_0_1km_polygon']
    did_local['pre_num_schools_1_2km']  = did_local['num_schools_1_2km_xy']
    did_local['post_num_schools_1_2km'] = did_local['num_schools_1_2km_polygon']

    did_local['pre_num_normal_schools_0_1km'] = (
        did_local['pre_num_schools_0_1km'] - did_local['pre_num_good_schools_0_1km']
    )
    did_local['post_num_normal_schools_0_1km'] = (
        did_local['post_num_schools_0_1km'] - did_local['post_num_good_schools_0_1km']
    )
    did_local['pre_num_normal_schools_1_2km'] = (
        did_local['pre_num_schools_1_2km'] - did_local['pre_num_good_schools_1_2km']
    )
    did_local['post_num_normal_schools_1_2km'] = (
        did_local['post_num_schools_1_2km'] - did_local['post_num_good_schools_1_2km']
    )

    # Keep normal-school composition fixed
    did_local = did_local[
        (did_local['pre_num_normal_schools_0_1km'] == did_local['post_num_normal_schools_0_1km']) &
        (did_local['pre_num_normal_schools_1_2km'] == did_local['post_num_normal_schools_1_2km'])
    ].copy()

    
    # -----------------------------------------------------------------------
    # 5e. CLEAN SCHOOL-SPECIFIC RESTRICTION
    # -----------------------------------------------------------------------
    # Treated units should gain the focal school as the second good school in 1-2km
    # Control units should continue to have exactly one good school in 1-2km
    did_local = did_local[
        ((did_local['treated'] == 1) & (did_local['pre_num_good_schools_1_2km'] == 1) & (did_local['post_num_good_schools_1_2km'] == 2)) |
        ((did_local['treated'] == 0) & (did_local['pre_num_good_schools_1_2km'] == 1) & (did_local['post_num_good_schools_1_2km'] == 1))
    ].copy()

    # -----------------------------------------------------------------------
    # 5f. FINAL REGRESSION VARIABLES
    # -----------------------------------------------------------------------
    did_local['year_month'] = did_local['Date'].dt.to_period('M').astype(str)
    did_local['log_price'] = np.log(did_local['resale_price'])
    did_local['period'] = np.where(did_local['Date'] >= post_start, 'post', 'pre')

    print(f"  Final local sample: {len(did_local)}")
    print(f"  Treated obs: {(did_local['treated'] == 1).sum()}")
    print(f"  Control obs: {(did_local['treated'] == 0).sum()}")

    if did_local.empty or did_local['treated'].nunique() < 2:
        print("  Skipping: no treated-control variation after filtering.")
        continue

    did_tight = did_local.copy()

    # ── 5g. PRINT COUNTS ──────────────────────────────────────────────────────
    counts = did_tight.groupby(['treated', 'period']).size().reset_index(name='n')
    print(f"\n  Transaction counts:")
    print(counts.to_string(index=False))

    # Skip if too few treated transactions
    treated_post = did_tight[(did_tight['treated'] == 1) & (did_tight['period'] == 'post')]
    treated_pre  = did_tight[(did_tight['treated'] == 1) & (did_tight['period'] == 'pre')]
    # if len(treated_post) < 10 or len(treated_pre) < 10:
    #     print(f"  Skipping — insufficient treated transactions (min 10 required per period)")
    #     continue

    # ── 5h. PARALLEL TRENDS TEST ──────────────────────────────────────────────
    did_tight['unit_group'] = 'ft_' + did_tight['flat_type'].astype(str) + "pre" + did_tight['good_school_names_1_2km_xy'].astype(str)

    numeric_covariates = [
        'floor_area_sqm',
        'remaining_lease',
        # 'num_nearby_mrt',
        # 'num_unique_mrt_lines',
        # 'num_nearby_malls',
    ]
    categorical_covariates = ['flat_model', 'storey_range']

    active_numeric_covariates = [
        col for col in numeric_covariates
        if did_tight[col].nunique(dropna=True) > 1
    ]
    active_categorical_covariates = [
        col for col in categorical_covariates
        if did_tight[col].nunique(dropna=True) > 1
    ]
    dropped_covariates = [
        col for col in numeric_covariates + categorical_covariates
        if col not in active_numeric_covariates + active_categorical_covariates
    ]

    print(f"  Dropping low-variation covariates: {dropped_covariates}")
    print(f"  Keeping numeric covariates: {active_numeric_covariates}")
    print(f"  Keeping categorical covariates: {active_categorical_covariates}")

    covariate_terms = active_numeric_covariates + [f"C({col})" for col in active_categorical_covariates]
    covariate_formula = f"+ {' + '.join(covariate_terms)}" if covariate_terms else ''

    pre_only  = did_tight[did_tight['period'] == 'pre'].copy()
    pre_only['year_quarter'] = pre_only['Date'].dt.to_period('Q').astype(str)
    # Then use C(year_quarter) instead of C(year_month) in the parallel trends test

    ref_month = sorted(pre_only['year_quarter'].unique())[-1]

    try:
        pt_model = smf.ols(
            f"""
            log_price ~ C(year_quarter, Treatment(reference='{ref_month}'))
            + treated:C(year_quarter, Treatment(reference='{ref_month}'))
            {covariate_formula}
            + C(unit_group)
            """,
            data=pre_only
        ).fit(cov_type='cluster', cov_kwds={'groups': pre_only['unit_group']})

        pt_terms = [t for t in pt_model.params.index if 'treated' in t and 'C(year_quarter' in t]
        if len(pt_terms) == 0:
            print(f"  Skipping — insufficient months for parallel trends test")
            continue

        R = np.zeros((len(pt_terms), len(pt_model.params)))
        for i, term in enumerate(pt_terms):
            R[i, pt_model.params.index.get_loc(term)] = 1
        joint_test = pt_model.f_test(R)
        pt_f   = float(joint_test.fvalue)
        pt_p   = float(joint_test.pvalue)
        print(f"\n  Parallel trends — F={pt_f:.4f}, p={pt_p:.4f} {'✅' if pt_p > 0.05 else '❌'}")

    except Exception as e:
        print(f"  Parallel trends test failed: {e}")
        pt_f, pt_p = np.nan, np.nan

    # ── 5j. MAIN DiD REGRESSION ───────────────────────────────────────────────
    try:
        did_model = smf.ols(
            f"""
            log_price ~ treated:post
            + C(year_month)
            {covariate_formula}
            + C(unit_group)
            """,
            data=did_tight
        ).fit(cov_type='cluster', cov_kwds={'groups': did_tight['unit_group']})

        coef    = did_model.params['treated:post']
        pval    = did_model.pvalues['treated:post']
        pct_eff = (np.exp(coef) - 1) * 100
        ci_low  = did_model.conf_int().loc['treated:post', 0]
        ci_high = did_model.conf_int().loc['treated:post', 1]

        def sig_stars(p):
            if p < 0.01:
                return '*** (1%)'
            elif p < 0.05:
                return '** (5%)'
            elif p < 0.10:
                return '* (10%)'
            else:
                return 'n.s.'

        print(f"\n  DiD estimate:   {coef:.4f}")
        print(f"  Exact % effect: {pct_eff:.2f}%")
        print(f"  95% CI:         [{(np.exp(ci_low)-1)*100:.2f}%, {(np.exp(ci_high)-1)*100:.2f}%]")
        print(f"  p-value:        {pval:.4f} {sig_stars(pval)}")

        results.append({
            'school_name':  school_name,
            'did_coef':     coef,
            'pct_effect':   pct_eff,
            'ci_low_pct':   (np.exp(ci_low)  - 1) * 100,
            'ci_high_pct':  (np.exp(ci_high) - 1) * 100,
            'pval':         pval,
            'pt_f':         pt_f,
            'pt_p':         pt_p,
            'paralleltrend_result':      '✅' if pt_p > 0.05 else '❌',  # added
            'did_result': '✅' if pval < 0.1 else '❌',  # added
            'n_treated_pre':  len(treated_pre),
            'n_treated_post': len(treated_post),
            'n_control_pre':  len(did_tight[(did_tight['treated']==0) & (did_tight['period']=='pre')]),
            'n_control_post': len(did_tight[(did_tight['treated']==0) & (did_tight['period']=='post')]),
        })

    except Exception as e:
        print(f"  DiD model failed: {e}")



Processing: AI TONG SCHOOL
  Final local sample: 404
  Treated obs: 4
  Control obs: 400

  Transaction counts:
 treated period   n
       0   post 113
       0    pre 287
       1   post   1
       1    pre   3
  Dropping low-variation covariates: []
  Keeping numeric covariates: ['floor_area_sqm', 'remaining_lease']
  Keeping categorical covariates: ['flat_model', 'storey_range']


c:\Users\rob-l\Documents\NUS\Y4S2\DSA4264\DSA4264-Project\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 13, but rank is 3
  warnings.warn('covariance of constraints does not have full '



  Parallel trends — F=221.3033, p=0.0000 ❌

  DiD estimate:   0.1984
  Exact % effect: 21.94%
  95% CI:         [11.35%, 33.55%]
  p-value:        0.0000 *** (1%)

Processing: ANGLO-CHINESE SCHOOL (JUNIOR)
  Final local sample: 243
  Treated obs: 0
  Control obs: 243
  Skipping: no treated-control variation after filtering.

Processing: ANGLO-CHINESE SCHOOL (PRIMARY)
  Final local sample: 319
  Treated obs: 0
  Control obs: 319
  Skipping: no treated-control variation after filtering.

Processing: CATHOLIC HIGH SCHOOL
  Final local sample: 901
  Treated obs: 170
  Control obs: 731

  Transaction counts:
 treated period   n
       0   post 224
       0    pre 507
       1   post  50
       1    pre 120
  Dropping low-variation covariates: []
  Keeping numeric covariates: ['floor_area_sqm', 'remaining_lease']
  Keeping categorical covariates: ['flat_model', 'storey_range']

  Parallel trends — F=322.8743, p=0.0000 ❌

  DiD estimate:   0.0317
  Exact % effect: 3.22%
  95% CI:         [-0

c:\Users\rob-l\Documents\NUS\Y4S2\DSA4264\DSA4264-Project\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 13, but rank is 12
  warnings.warn('covariance of constraints does not have full '


  Final local sample: 324
  Treated obs: 40
  Control obs: 284

  Transaction counts:
 treated period   n
       0   post 119
       0    pre 165
       1   post  18
       1    pre  22
  Dropping low-variation covariates: []
  Keeping numeric covariates: ['floor_area_sqm', 'remaining_lease']
  Keeping categorical covariates: ['flat_model', 'storey_range']

  Parallel trends — F=571.8412, p=0.0000 ❌

  DiD estimate:   0.0825
  Exact % effect: 8.60%
  95% CI:         [2.85%, 14.68%]
  p-value:        0.0030 *** (1%)

Processing: CHIJ ST. NICHOLAS GIRLS' SCHOOL


...

  Final local sample: 216
  Treated obs: 18
  Control obs: 198

  Transaction counts:
 treated period   n
       0   post  64
       0    pre 134
       1   post   7
       1    pre  11
  Dropping low-variation covariates: []
  Keeping numeric covariates: ['floor_area_sqm', 'remaining_lease']
  Keeping categorical covariates: ['flat_model', 'storey_range']

  Parallel trends — F=19.1673, p=0.0185 ❌

  DiD estimate:   -0.0151
  Exact % effect: -1.50%
  95% CI:         [-3.38%, 0.42%]
  p-value:        0.1259 n.s.

Processing: ST. HILDA'S PRIMARY SCHOOL


c:\Users\rob-l\Documents\NUS\Y4S2\DSA4264\DSA4264-Project\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 13, but rank is 3
  warnings.warn('covariance of constraints does not have full '


  Final local sample: 809
  Treated obs: 0
  Control obs: 809
  Skipping: no treated-control variation after filtering.

Processing: ST. JOSEPH'S INSTITUTION JUNIOR
  Final local sample: 94
  Treated obs: 37
  Control obs: 57

  Transaction counts:
 treated period  n
       0   post 22
       0    pre 35
       1   post 18
       1    pre 19
  Dropping low-variation covariates: []
  Keeping numeric covariates: ['floor_area_sqm', 'remaining_lease']
  Keeping categorical covariates: ['flat_model', 'storey_range']

  Parallel trends — F=3.4369, p=0.1688 ✅

  DiD estimate:   0.0247
  Exact % effect: 2.50%
  95% CI:         [-2.50%, 7.76%]
  p-value:        0.3334 n.s.

Processing: TAO NAN SCHOOL


c:\Users\rob-l\Documents\NUS\Y4S2\DSA4264\DSA4264-Project\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 13, but rank is 3
  warnings.warn('covariance of constraints does not have full '
c:\Users\rob-l\Documents\NUS\Y4S2\DSA4264\DSA4264-Project\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 13, but rank is 9
  warnings.warn('covariance of constraints does not have full '


  Final local sample: 509
  Treated obs: 48
  Control obs: 461

  Transaction counts:
 treated period   n
       0   post 194
       0    pre 267
       1   post  19
       1    pre  29
  Dropping low-variation covariates: []
  Keeping numeric covariates: ['floor_area_sqm', 'remaining_lease']
  Keeping categorical covariates: ['flat_model', 'storey_range']

  Parallel trends — F=5062.9850, p=0.0000 ❌

  DiD estimate:   0.1392
  Exact % effect: 14.94%
  95% CI:         [2.36%, 29.07%]
  p-value:        0.0186 ** (5%)


## Cross-School Summary Table

The last active cell stacks the per-school estimates into a single summary table so the restrictive second-school specification can be reviewed school by school.

In [13]:
# ── 6. SUMMARY TABLE ──────────────────────────────────────────────────────────
results_df = pd.DataFrame(results).sort_values('pt_p', ascending=False)
print(f"\n\n{'='*60}")
print("SUMMARY OF RESULTS ACROSS ALL SCHOOLS")
print(f"{'='*60}")
print(results_df[[
    'school_name', 'did_coef', 'pct_effect', 'pt_p', 'paralleltrend_result', 
    'pval', 'did_result',
    'n_treated_pre', 'n_treated_post', 'n_control_pre', 'n_control_post'
]].to_string(index=False))

#results_df.to_csv("../outputs/did_results_by_school.csv", index=False)
#print("\nResults saved to ../outputs/did_results_by_school.csv")




SUMMARY OF RESULTS ACROSS ALL SCHOOLS
                          school_name  did_coef  pct_effect         pt_p paralleltrend_result         pval did_result  n_treated_pre  n_treated_post  n_control_pre  n_control_post
      ST. JOSEPH'S INSTITUTION JUNIOR  0.024695    2.500267 1.688233e-01                    ✅ 3.334344e-01          ❌             19              18             35              22
               NANYANG PRIMARY SCHOOL  0.060360    6.221935 3.127674e-02                    ❌ 3.131913e-17          ✅            184              87            293             117
                RULANG PRIMARY SCHOOL  0.003362    0.336806 2.131179e-02                    ❌ 9.261651e-01          ❌             22              11            227             118
            SOUTH VIEW PRIMARY SCHOOL -0.015066   -1.495338 1.846329e-02                    ❌ 1.258793e-01          ❌             11               7            134              64
       HOLY INNOCENTS' PRIMARY SCHOOL  0.024678    2.498455 

## Archived SDID / Imputation Section

The remaining cells are kept only as archived exploratory material.

They document older Synthetic DiD and imputation workflows that are currently unused in the unpooled notebook.

In [54]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "models" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from analysis.diffdiff_notebook_helpers import (
    fit_prepared_diffdiff_panel,
    prepare_no_imputation_panel,
    prepare_half_year_panel,
    print_sdid_weight_tables,
)
def run_diffdiff_balance_method(obj, balance_method, label):
    school_name = obj['school_name']
    sdid_panel = obj['sdid_panel'].copy()
    covariate_cols = obj['covariate_cols']

    print()
    print('=' * 70)
    print(f'{label}: {school_name}')
    print('=' * 70)

    if balance_method == 'inner':
        prep = prepare_no_imputation_panel(sdid_panel, covariate_cols, balance_method='inner')
        print("  Workflow: inner balance without imputation")
    elif balance_method == 'half_year':
        prep = prepare_half_year_panel(sdid_panel, covariate_cols, balance_method='inner')
        print("  Workflow: half-year imputation before inner balancing")
        print(f"  Rows imputed from same half-year: {len(prep['imputed_rows_df'])}")
        if not prep['imputed_rows_df'].empty:
            print(prep['imputed_rows_df'].to_string(index=False))
    else:
        raise ValueError(f"Unsupported balance_method: {balance_method}")
    fit = fit_prepared_diffdiff_panel(prep['panel_balanced'], covariate_cols)

    if fit['skip_reason'] is not None:
        print(f"  Skipping - {fit['skip_reason']}")
        if 'n_pre_periods' in fit and 'n_post_periods' in fit:
            print(f"  Pre periods after balancing: {fit['n_pre_periods']}")
            print(f"  Post periods after balancing: {fit['n_post_periods']}")
        if 'all_periods' in fit:
            print(f"  Periods retained after balancing: {fit['all_periods']}")
        return None

    print(f"  Units after balancing: {fit['n_treated_units'] + fit['n_control_units']}")
    print(f"  Periods after balancing: {fit['n_periods']}")
    print(f"  Treated units after balancing: {fit['n_treated_units']}")
    print(f"  Control units after balancing: {fit['n_control_units']}")
    if fit['dropped_covariates']:
        print(f"  Covariates excluded due to NA after balancing: {fit['dropped_covariates']}")
    print('  Synthetic DID fit completed.')
    print(f"  ATT (log points): {fit['att']:.4f}")
    print(f"  Approx % effect:  {fit['pct_effect']:.2f}%")
    print(f"  p-value:          {fit['p_value']:.4f}")
    print_sdid_weight_tables(fit['unit_weights_df'], fit['time_weights_df'])

    return {
        'school_name': school_name,
        'att': fit['att'],
        'se': fit['se'],
        'pct_effect': fit['pct_effect'],
        'p_value': fit['p_value'],
        'conf_int': fit['conf_int'],
        'n_treated_units': fit['n_treated_units'],
        'n_control_units': fit['n_control_units'],
        'n_periods': fit['n_periods'],
        'covariates_used': fit['fit_covariates'],
        'imputation_mode': balance_method,
        'imputed_rows': len(prep['imputed_rows_df']),
        'sdid_result': 'PASS' if fit['p_value'] < 0.10 else 'FAIL',
        'res': fit['res'],
        'panel_balanced': fit['panel_balanced'].copy(),
        'post_periods': fit['post_periods'],
        'unit_weights_df': fit['unit_weights_df'].copy(),
        'time_weights_df': fit['time_weights_df'].copy(),
    }


In [55]:
synthetic_did_inputs = []

for school_name in results_df['school_name'].tolist():
    print()
    print('=' * 70)
    print(f'Synthetic DID panel build: {school_name}')
    print('=' * 70)

    school_match = good_school_poly_gdf[
        good_school_poly_gdf['school_name'].str.strip().str.upper() == school_name.strip().upper()
    ]
    if school_match.empty:
        print('  Skipping - school polygon not found')
        continue

    school = school_match.iloc[0]
    school_geom = school['geometry']
    buffer_2km = school_geom.buffer(2000)
    buffer_3km = school_geom.buffer(3000)
    hdb_local = resale_gdf[resale_gdf.geometry.within(buffer_3km)].copy()
    if hdb_local.empty:
        print('  Skipping - no local transactions within 3km')
        continue

    hdb_local = hdb_local[
        ((hdb_local['Date'] >= pre_start) & (hdb_local['Date'] <= pre_end))
        | ((hdb_local['Date'] >= post_start) & (hdb_local['Date'] <= post_end))
    ].copy()
    if hdb_local.empty:
        print('  Skipping - no transactions in sample window')
        continue

    hdb_local['post'] = (hdb_local['Date'] >= post_start).astype(int)
    hdb_local['period'] = np.where(hdb_local['Date'] >= post_start, 'post', 'pre')
    hdb_local['year_quarter'] = hdb_local['Date'].dt.to_period('Q').astype(str)
    hdb_local['log_price'] = np.log(hdb_local['resale_price'])
    hdb_local['in_2_3km_band'] = (
        hdb_local.geometry.within(buffer_3km)
        & ~hdb_local.geometry.within(buffer_2km)
    ).astype(int)

    hdb_local['pre_in_0_1km_focal'] = hdb_local['good_school_names_0_1km_xy'].apply(
        lambda x: has_focal_school(x, school_name)
    ).astype(int)
    hdb_local['post_in_0_1km_focal'] = hdb_local['good_school_names_0_1km_polygon'].apply(
        lambda x: has_focal_school(x, school_name)
    ).astype(int)

    hdb_local['pre_in_1_2km_focal'] = hdb_local['good_school_names_1_2km_xy'].apply(
        lambda x: has_focal_school(x, school_name)
    ).astype(int)
    hdb_local['post_in_1_2km_focal'] = hdb_local['good_school_names_1_2km_polygon'].apply(
        lambda x: has_focal_school(x, school_name)
    ).astype(int)

    did_local = hdb_local[
        (hdb_local['pre_in_0_1km_focal'] == 0)
        & (hdb_local['pre_in_1_2km_focal'] == 0)
    ].copy()
    did_local['treated'] = (
        (did_local['pre_in_0_1km_focal'] == 0)
        & (did_local['post_in_0_1km_focal'] == 0)
        & (did_local['pre_in_1_2km_focal'] == 0)
        & (did_local['post_in_1_2km_focal'] == 1)
    ).astype(int)
    did_local = did_local[
        ((did_local['treated'] == 1) & (did_local['post_in_0_1km_focal'] == 0) & (did_local['post_in_1_2km_focal'] == 1))
        | ((did_local['treated'] == 0) & (did_local['in_2_3km_band'] == 1) & (did_local['post_in_0_1km_focal'] == 0) & (did_local['post_in_1_2km_focal'] == 0))
    ].copy()
    if did_local.empty:
        print('  Skipping - empty did_local after focal-school assignment')
        continue

    did_local['pre_num_good_schools_0_1km'] = did_local['num_good_schools_0_1km_xy']
    did_local['post_num_good_schools_0_1km'] = did_local['num_good_schools_0_1km_polygon']
    did_local['pre_num_good_schools_1_2km'] = did_local['num_good_schools_1_2km_xy']
    did_local['post_num_good_schools_1_2km'] = did_local['num_good_schools_1_2km_polygon']
    did_local['pre_num_schools_0_1km'] = did_local['num_schools_0_1km_xy']
    did_local['post_num_schools_0_1km'] = did_local['num_schools_0_1km_polygon']
    did_local['pre_num_schools_1_2km'] = did_local['num_schools_1_2km_xy']
    did_local['post_num_schools_1_2km'] = did_local['num_schools_1_2km_polygon']
    did_local['pre_num_normal_schools_0_1km'] = (
        did_local['pre_num_schools_0_1km'] - did_local['pre_num_good_schools_0_1km']
    )
    did_local['post_num_normal_schools_0_1km'] = (
        did_local['post_num_schools_0_1km'] - did_local['post_num_good_schools_0_1km']
    )
    did_local['pre_num_normal_schools_1_2km'] = (
        did_local['pre_num_schools_1_2km'] - did_local['pre_num_good_schools_1_2km']
    )
    did_local['post_num_normal_schools_1_2km'] = (
        did_local['post_num_schools_1_2km'] - did_local['post_num_good_schools_1_2km']
    )

    did_local = did_local[
        (did_local['pre_num_normal_schools_0_1km'] == did_local['post_num_normal_schools_0_1km'])
        & (did_local['pre_num_normal_schools_1_2km'] == did_local['post_num_normal_schools_1_2km'])
    ].copy()
    did_local = did_local[
        ((did_local['treated'] == 1) & (did_local['pre_num_good_schools_1_2km'] == 1) & (did_local['post_num_good_schools_1_2km'] == 2))
        | ((did_local['treated'] == 0) & (did_local['pre_num_good_schools_1_2km'] == 1) & (did_local['post_num_good_schools_1_2km'] == 1))
    ].copy()

    print(f'  Final local sample: {len(did_local)}')
    print(f"  Treated obs: {(did_local['treated'] == 1).sum()}")
    print(f"  Control obs: {(did_local['treated'] == 0).sum()}")

    if did_local.empty or did_local['treated'].nunique() < 2:
        print('  Skipping - no treated-control variation after filtering')
        continue

    did_local['unit_id'] = (
        np.where(did_local['treated'] == 1, 'TREATED_', 'CONTROL_')
        + did_local['flat_type'].astype(str).str.strip()
    )

    numeric_covariate_cols = [
        'floor_area_sqm',
        'remaining_lease',
        'num_unique_mrt_lines',
        'num_nearby_malls',
    ]
    numeric_covariate_cols = [c for c in numeric_covariate_cols if c in did_local.columns]

    # Use categorical dummy variables (k-1 coding), not full one-hot encoding.
    categorical_dummies = pd.get_dummies(
        did_local[['flat_model', 'storey_range']].fillna('MISSING'),
        prefix=['flat_model', 'storey_range'],
        prefix_sep='__',
        drop_first=True,
        dtype=float,
    )
    categorical_dummies.columns = [
        c.replace(' ', '_').replace('/', '_').replace('-', '_')
        for c in categorical_dummies.columns
    ]
    did_local = pd.concat([did_local, categorical_dummies], axis=1)
    categorical_covariate_cols = categorical_dummies.columns.tolist()

    covariate_cols = numeric_covariate_cols + categorical_covariate_cols

    agg_spec = {
        'treated': ('treated', 'max'),
        'post': ('post', 'max'),
        'log_price': ('log_price', 'mean'),
    }
    for c in covariate_cols:
        agg_spec[c] = (c, 'mean')

    sdid_panel = (
        did_local.groupby(['unit_id', 'year_quarter'], as_index=False)
        .agg(**agg_spec)
    )
    if sdid_panel.empty:
        print('  Skipping - empty sdid_panel')
        continue

    print(f"  Numeric SDID covariates: {numeric_covariate_cols}")
    print(f"  Categorical SDID dummy covariates added: {len(categorical_covariate_cols)} columns")
    print(f"  Raw SDID panel rows: {len(sdid_panel)}")
    print(f"  Units before balancing: {sdid_panel['unit_id'].nunique()}")
    print(f"  Periods before balancing: {sdid_panel['year_quarter'].nunique()}")

    all_units = sorted(sdid_panel['unit_id'].unique())
    all_periods = sorted(sdid_panel['year_quarter'].unique())
    expected_rows = len(all_units) * len(all_periods)
    actual_rows = len(sdid_panel)
    missing_rows = expected_rows - actual_rows

    print(f"  Expected unit-quarter rows: {expected_rows}")
    print(f"  Actual unit-quarter rows:   {actual_rows}")
    print(f"  Missing unit-quarter rows:  {missing_rows}")

    synthetic_did_inputs.append({
        'school_name': school_name,
        'sdid_panel': sdid_panel.copy(),
        'covariate_cols': covariate_cols.copy(),
    })

print()
print(f'Total schools prepared for SDID balance-method runs: {len(synthetic_did_inputs)}')



Synthetic DID panel build: NAN CHIAU PRIMARY SCHOOL
  Final local sample: 1899
  Treated obs: 203
  Control obs: 1696
  Numeric SDID covariates: ['floor_area_sqm', 'remaining_lease', 'num_unique_mrt_lines', 'num_nearby_malls']
  Categorical SDID dummy covariates added: 14 columns
  Raw SDID panel rows: 152
  Units before balancing: 9
  Periods before balancing: 19
  Expected unit-quarter rows: 171
  Actual unit-quarter rows:   152
  Missing unit-quarter rows:  19

Synthetic DID panel build: CATHOLIC HIGH SCHOOL
  Final local sample: 449
  Treated obs: 158
  Control obs: 291
  Numeric SDID covariates: ['floor_area_sqm', 'remaining_lease', 'num_unique_mrt_lines', 'num_nearby_malls']
  Categorical SDID dummy covariates added: 13 columns
  Raw SDID panel rows: 96
  Units before balancing: 6
  Periods before balancing: 19
  Expected unit-quarter rows: 114
  Actual unit-quarter rows:   96
  Missing unit-quarter rows:  18

Synthetic DID panel build: CHONGFU SCHOOL
  Final local sample: 1446


In [56]:
synthetic_did_results_inner = []
synthetic_did_results_half_year = []

for obj in synthetic_did_inputs:
    result = run_diffdiff_balance_method(obj, 'inner', 'Inner-balance SDID')
    if result is not None:
        synthetic_did_results_inner.append(result)

for obj in synthetic_did_inputs:
    result = run_diffdiff_balance_method(obj, 'half_year', 'Half-year imputation SDID')
    if result is not None:
        synthetic_did_results_half_year.append(result)

synthetic_did_results_df = pd.DataFrame(synthetic_did_results_inner + synthetic_did_results_half_year)
if synthetic_did_results_df.empty:
    print('No SDID results were produced.')
else:
    display(
        synthetic_did_results_df[[
            'school_name',
            'imputation_mode',
            'att',
            'pct_effect',
            'p_value',
            'sdid_result',
            'n_treated_units',
            'n_control_units',
            'n_periods',
            'imputed_rows',
        ]].sort_values(['school_name', 'imputation_mode'])
    )



Inner-balance SDID: NAN CHIAU PRIMARY SCHOOL
  Workflow: inner balance without imputation
  Skipping - balanced panel lacks treated or control units

Inner-balance SDID: CATHOLIC HIGH SCHOOL
  Workflow: inner balance without imputation
  Skipping - balanced panel lacks treated or control units

Inner-balance SDID: CHONGFU SCHOOL
  Workflow: inner balance without imputation
  Skipping - balanced panel lacks treated or control units

Inner-balance SDID: HOLY INNOCENTS' PRIMARY SCHOOL
  Workflow: inner balance without imputation
  Skipping - balanced panel lacks treated or control units

Inner-balance SDID: KONG HWA SCHOOL
  Workflow: inner balance without imputation
  Skipping - balanced panel lacks treated or control units

Inner-balance SDID: ST. JOSEPH'S INSTITUTION JUNIOR
  Workflow: inner balance without imputation
  Skipping - balanced panel lacks treated or control units

Inner-balance SDID: FRONTIER PRIMARY SCHOOL
  Workflow: inner balance without imputation
  Skipping - balance

c:\Users\rob-l\Documents\NUS\Y4S2\DSA4264\DSA4264-Project\.venv\Lib\site-packages\diff_diff\synthetic_did.py:539: UserWarning: Rank-deficient design matrix: dropping 2 of 52 columns (column 5, column 8). Coefficients for these columns are set to NA. This may indicate multicollinearity in your model specification.
  coeffs, residuals, _ = solve_ols(X_full, y, return_vcov=False)
C:\Users\rob-l\Documents\NUS\Y4S2\DSA4264\DSA4264-Project\analysis\diffdiff_notebook_helpers.py:107: UserWarning: Pre-treatment fit is poor: RMSE (0.0179) exceeds the standard deviation of treated pre-treatment outcomes (0.0113). The synthetic control may not adequately reproduce treated unit trends. Consider adding more control units or adjusting regularization.
  result = sdid.fit(**fit_kwargs)


  Workflow: half-year imputation before inner balancing
  Rows imputed from same half-year: 13
       unit_id year_quarter year_half
CONTROL_2 ROOM       2020Q1   2020-H1
CONTROL_2 ROOM       2020Q3   2020-H2
CONTROL_2 ROOM       2021Q2   2021-H1
CONTROL_2 ROOM       2021Q3   2021-H2
CONTROL_2 ROOM       2022Q1   2022-H1
CONTROL_2 ROOM       2022Q3   2022-H2
TREATED_2 ROOM       2018Q3   2018-H2
TREATED_2 ROOM       2022Q4   2022-H2
TREATED_3 ROOM       2020Q4   2020-H2
TREATED_4 ROOM       2020Q2   2020-H1
TREATED_4 ROOM       2020Q3   2020-H2
TREATED_5 ROOM       2019Q1   2019-H1
TREATED_5 ROOM       2021Q1   2021-H1
  Units after balancing: 8
  Periods after balancing: 19
  Treated units after balancing: 3
  Control units after balancing: 5
  Synthetic DID fit completed.
  ATT (log points): 0.0056
  Approx % effect:  0.57%
  p-value:          0.8200
  Unit weights:
             unit   weight
   CONTROL_5 ROOM 0.216461
   CONTROL_2 ROOM 0.205989
   CONTROL_4 ROOM 0.199783
CONTROL_EXE

C:\Users\rob-l\Documents\NUS\Y4S2\DSA4264\DSA4264-Project\analysis\diffdiff_notebook_helpers.py:107: UserWarning: Pre-treatment fit is poor: RMSE (0.0163) exceeds the standard deviation of treated pre-treatment outcomes (0.0108). The synthetic control may not adequately reproduce treated unit trends. Consider adding more control units or adjusting regularization.
  result = sdid.fit(**fit_kwargs)


  Workflow: half-year imputation before inner balancing
  Rows imputed from same half-year: 11
       unit_id year_quarter year_half
CONTROL_4 ROOM       2019Q2   2019-H1
CONTROL_4 ROOM       2019Q4   2019-H2
CONTROL_4 ROOM       2020Q3   2020-H2
CONTROL_5 ROOM       2020Q3   2020-H2
CONTROL_5 ROOM       2022Q4   2022-H2
TREATED_3 ROOM       2019Q2   2019-H1
TREATED_3 ROOM       2019Q4   2019-H2
TREATED_3 ROOM       2021Q2   2021-H1
TREATED_3 ROOM       2022Q1   2022-H1
TREATED_4 ROOM       2020Q2   2020-H1
TREATED_5 ROOM       2019Q2   2019-H1
  Units after balancing: 4
  Periods after balancing: 19
  Treated units after balancing: 2
  Control units after balancing: 2
  Synthetic DID fit completed.
  ATT (log points): -0.0568
  Approx % effect:  -5.52%
  p-value:          nan
  Unit weights:
          unit   weight
CONTROL_3 ROOM 0.579348
CONTROL_4 ROOM 0.420652
  Time weights:
period       weight
2020Q1 8.895489e-01
2021Q1 1.104511e-01
2019Q4 1.228237e-10
2018Q4 0.000000e+00
2018Q3 0

...

C:\Users\rob-l\Documents\NUS\Y4S2\DSA4264\DSA4264-Project\analysis\diffdiff_notebook_helpers.py:107: UserWarning: Pre-treatment fit is poor: RMSE (0.0192) exceeds the standard deviation of treated pre-treatment outcomes (0.0118). The synthetic control may not adequately reproduce treated unit trends. Consider adding more control units or adjusting regularization.
  result = sdid.fit(**fit_kwargs)


  Workflow: half-year imputation before inner balancing
  Rows imputed from same half-year: 26
          unit_id year_quarter year_half
   CONTROL_2 ROOM       2019Q1   2019-H1
   CONTROL_2 ROOM       2019Q4   2019-H2
   CONTROL_2 ROOM       2020Q2   2020-H1
   CONTROL_2 ROOM       2022Q4   2022-H2
CONTROL_EXECUTIVE       2018Q3   2018-H2
CONTROL_EXECUTIVE       2020Q2   2020-H1
   TREATED_4 ROOM       2019Q2   2019-H1
   TREATED_4 ROOM       2019Q3   2019-H2
   TREATED_4 ROOM       2020Q3   2020-H2
   TREATED_4 ROOM       2021Q4   2021-H2
   TREATED_4 ROOM       2022Q1   2022-H1
   TREATED_4 ROOM       2022Q3   2022-H2
   TREATED_5 ROOM       2018Q3   2018-H2
   TREATED_5 ROOM       2019Q2   2019-H1
   TREATED_5 ROOM       2020Q2   2020-H1
   TREATED_5 ROOM       2020Q4   2020-H2
   TREATED_5 ROOM       2021Q1   2021-H1
   TREATED_5 ROOM       2021Q4   2021-H2
   TREATED_5 ROOM       2022Q2   2022-H1
   TREATED_5 ROOM       2022Q4   2022-H2
TREATED_EXECUTIVE       2019Q2   2019-H1
TRE

c:\Users\rob-l\Documents\NUS\Y4S2\DSA4264\DSA4264-Project\.venv\Lib\site-packages\diff_diff\synthetic_did.py:539: UserWarning: Rank-deficient design matrix: dropping 5 of 31 columns (column 0, column 3, column 5, column 11, column 12). Coefficients for these columns are set to NA. This may indicate multicollinearity in your model specification.
  coeffs, residuals, _ = solve_ols(X_full, y, return_vcov=False)
C:\Users\rob-l\Documents\NUS\Y4S2\DSA4264\DSA4264-Project\analysis\diffdiff_notebook_helpers.py:107: UserWarning: Pre-treatment fit is poor: RMSE (0.0375) exceeds the standard deviation of treated pre-treatment outcomes (0.0196). The synthetic control may not adequately reproduce treated unit trends. Consider adding more control units or adjusting regularization.
  result = sdid.fit(**fit_kwargs)
C:\Users\rob-l\Documents\NUS\Y4S2\DSA4264\DSA4264-Project\analysis\diffdiff_notebook_helpers.py:107: UserWarning: Not enough control units (1) for placebo variance estimation with 1 treate

  Workflow: half-year imputation before inner balancing
  Rows imputed from same half-year: 13
       unit_id year_quarter year_half
CONTROL_2 ROOM       2022Q1   2022-H1
CONTROL_3 ROOM       2022Q1   2022-H1
CONTROL_4 ROOM       2019Q2   2019-H1
CONTROL_4 ROOM       2021Q2   2021-H1
CONTROL_5 ROOM       2020Q3   2020-H2
CONTROL_5 ROOM       2022Q1   2022-H1
TREATED_3 ROOM       2019Q1   2019-H1
TREATED_3 ROOM       2020Q3   2020-H2
TREATED_3 ROOM       2021Q2   2021-H1
TREATED_3 ROOM       2021Q3   2021-H2
TREATED_4 ROOM       2018Q3   2018-H2
TREATED_4 ROOM       2019Q2   2019-H1
TREATED_4 ROOM       2022Q2   2022-H1
  Skipping - empty balanced panel

Half-year imputation SDID: SINGAPORE CHINESE GIRLS' PRIMARY SCHOOL
  Workflow: half-year imputation before inner balancing
  Rows imputed from same half-year: 8
       unit_id year_quarter year_half
CONTROL_3 ROOM       2019Q4   2019-H2
CONTROL_4 ROOM       2019Q2   2019-H1
CONTROL_4 ROOM       2022Q1   2022-H1
CONTROL_4 ROOM       2022

,school_name,imputation_mode,att,pct_effect,p_value,sdid_result,n_treated_units,n_control_units,n_periods,imputed_rows
6,AI TONG SCHOOL,half_year,0.049647,5.090038,0.004975,PASS,1,2,19,19
2,CATHOLIC HIGH SCHOOL,half_year,-0.056818,-5.523386,NaN,FAIL,2,2,19,11
12,CHIJ ST. NICHOLAS GIRLS' SCHOOL,half_year,-0.007566,-0.753745,NaN,FAIL,1,1,18,9
10,FAIRFIELD METHODIST SCHOOL (PRIMARY),half_year,0.010562,1.061792,1.000000,FAIL,2,3,19,15
0,FAIRFIELD METHODIST SCHOOL (PRIMARY),inner,0.010562,1.061792,1.000000,FAIL,2,3,19,0
5,FRONTIER PRIMARY SCHOOL,half_year,0.074081,7.689382,0.004975,PASS,1,3,19,6
3,HOLY INNOCENTS' PRIMARY SCHOOL,half_year,-0.001428,-0.142668,0.735000,FAIL,1,4,19,8
1,NAN CHIAU PRIMARY SCHOOL,half_year,0.005638,0.565400,0.820000,FAIL,3,5,19,13
7,PRINCESS ELIZABETH PRIMARY SCHOOL,half_year,0.016700,1.683972,0.570000,FAIL,2,5,19,15
8,ROSYTH SCHOOL,half_year,0.027240,2.761490,0.270000,FAIL,1,4,19,19
